# RTO Module Maintenance Cost Optimizer — Full Pipeline Demo

**End-to-end story**: from raw metrology data → wafer-level uniformity → lot-level SPC → PM deferral decision.

This notebook demonstrates the full monitoring pipeline:

1. **Generate** hierarchical synthetic data (60 lots × 5 wafers × 49 sites)
2. **Inspect** a wafer map (within-wafer uniformity)
3. **Aggregate** site data into lot-level X̄-S chart subgroups
4. **Visualize** the SPC monitoring dashboard
5. **Drill down** into the excursion lot via its wafer map
6. **Decide** whether scheduled PM can be deferred

All synthetic — no proprietary data is used.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from rta_optimizer.data_generator import (
    simulate_lot_history, aggregate_to_lot_subgroups, compute_wiw_trend
)
from rta_optimizer.wafer_map import (
    generate_49site_pattern, simulate_oxide_thickness, plot_wafer_map
)
from rta_optimizer.oxide_spc import (
    compute_xbar_s, plot_xbar_s, plot_full_monitoring_dashboard
)
from rta_optimizer.pm_deferral import (
    ToolState, DeferralConfig, evaluate_deferral, Decision
)

print('Pipeline modules loaded')

## 1. Generate Hierarchical Metrology Data

Simulating 60 lots from an RTO toolset. Each lot contains 5 wafers, each measured at 49 sites by an ellipsometer (KLA Aleris / Nanometrics Atlas-class).

**Variation hierarchy** (matches real fab):
- **Lot-to-lot**: Slow drift from lamp aging (~0.012 Å/lot) + random process variation
- **Wafer-to-wafer**: Within-lot variation from handling, position
- **Site-to-site**: Within-wafer (WIW) spatial signature + measurement noise

**Excursion injected** at lot 52 to validate Western Electric rule detection.

In [ ]:
thickness_3d, sites_xy = simulate_lot_history(
    n_lots=60,
    wafers_per_lot=5,
    sites_per_wafer=49,
    target=30.0,                   # gate oxide target (A)
    drift_per_lot=0.012,           # lamp aging
    inject_excursion_at=52,        # late-life excursion event
    excursion_magnitude=1.2,
    seed=42,
)

print(f'Data shape: {thickness_3d.shape}')
print(f'             = (n_lots, n_wafers, n_sites)')
print(f'Total measurements: {thickness_3d.size:,}')
print(f'Grand mean: {thickness_3d.mean():.3f} A')
print(f'Grand sigma: {thickness_3d.std():.3f} A')

## 2. Wafer-Level Inspection: Single Wafer Map

Before aggregating, let's inspect a wafer's within-wafer uniformity. Pick the first wafer of lot 5 (healthy region) to see the baseline spatial signature.

In [ ]:
fig, stats = plot_wafer_map(
    sites_xy, thickness_3d[5, 0],
    target=30.0, USL=31.5, LSL=28.5,
    title='Wafer Map - Lot 5, Wafer 1 (healthy baseline)',
)
plt.show()

print(f'WIW 1-sigma uniformity: {stats["NU_1sigma_pct"]:.2f}%')
print(f'Per-wafer quality margin index: {stats["quality margin index"]:.2f}')

## 3. Aggregate Site Data to Lot-Level SPC Subgroups

Standard semiconductor SPC practice:
- Each **wafer** is summarized by its **mean Tox** (avg of 49 sites)
- Each **lot** provides a **subgroup of 5 wafer means** for X̄-S charts
- WIW variability (49 sites) is tracked separately as NU% — not lumped into SPC σ (sites are spatially correlated, would violate i.i.d. assumption)

In [ ]:
# Site -> wafer mean -> lot subgroup
subgroups = aggregate_to_lot_subgroups(thickness_3d)
print(f'Subgroup matrix: {subgroups.shape}  = (n_lots, n_wafers)')
print(f'First 3 lots, wafer means:')
print(subgroups[:3].round(3))

In [ ]:
# Apply X-bar S charts with spec limits
result = compute_xbar_s(subgroups, USL=31.5, LSL=28.5)

print(f'X-bar-bar:  {result.X_bar_bar:.3f} A')
print(f'S-bar:      {result.S_bar:.3f} A')
print(f'Sigma-hat:  {result.sigma_hat:.3f} A')
print(f'quality margin index:        {result.quality margin index:.2f}')
print(f'WE rule violations (unique lots): '
      f'{len(set(v[0] for v in result.violations))}')

## 4. Full Monitoring Dashboard

The 4-panel dashboard is the **single visual** a process engineer would review daily:

- **X̄ Chart** (top-left): Lot mean tracking with control + spec limits, WE violations boxed
- **S Chart** (top-right): Wafer-to-wafer variability per lot
- **WIW NU% Trend** (bottom-left): Within-wafer non-uniformity over time — proxy for lamp degradation
- **Rolling quality margin index** (bottom-right): Guardband trend with windowed quality margin index (n=10 lots)

In [ ]:
fig, dash_result = plot_full_monitoring_dashboard(
    thickness_3d, USL=31.5, LSL=28.5, target=30.0,
)
plt.show()

**Observations**:
- Clear upward drift in X̄ chart (lamp aging signature)
- Excursion at lot 52 visible as a spike beyond UCL
- Rolling quality margin index degrades from ~4.5 (early life) toward ~3.0 (late life)
- WIW NU% stays below 1% but trends upward → signals tool drift before catastrophic excursion

## 5. Excursion Drill-Down: Wafer Map of Lot 52

When the SPC dashboard flags a violation, the standard response is to drill into the wafer map for **root cause inference** via signature pattern.

In [ ]:
fig, exc_stats = plot_wafer_map(
    sites_xy, thickness_3d[52, 0],
    target=30.0, USL=31.5, LSL=28.5,
    title='EXCURSION - Lot 52, Wafer 1 (bullseye signature)',
)
plt.show()

print(f'Excursion wafer mean: {exc_stats["mean"]:.3f} A (target=30.0)')
print(f'WIW 1-sigma: {exc_stats["sigma"]:.3f} A')
print(f'quality margin index: {exc_stats["quality margin index"]:.2f}')

if exc_stats.get('oos_sites', 0) > 0:
    print(f'OOS sites: {exc_stats["oos_sites"]} - some sites exceed spec')

**Root cause hypothesis** (based on signature):
- Center thicker than edge → **bullseye pattern**
- Consistent with **lamp degradation** (radial thermal profile non-uniformity)
- Action: schedule emergency PM, replace lamp module

## 6. PM Deferral Decision Engine

When the tool reaches scheduled PM (say day 27 of a 30-day cycle), the engine evaluates whether maintenance can be **safely deferred** by 5 days.

**Decision logic** (priority order):

1. **Hard stops**: SPC violation? quality margin index<1.33? Beyond max deferral cap? → EMERGENCY or PROCEED
2. **Risk score** (0-100%): Weibull hazard + drift projection + quality margin index margin
3. **Economic check**: PM cost saved by deferral vs risk-weighted excursion cost
4. **Final**: DEFER only if risk < 35% AND net benefit > 0

### 6.1 Scenario A — Healthy tool, mid-cycle (expect DEFER)

In [ ]:
state_a = ToolState(
    days_since_last_pm=15,
    scheduled_pm_interval=45,
    recent_quality_index=1.85,
    spc_violations_last_7d=0,
    drift_slope_per_day=0.005,
    sigma_margin_to_USL=3.5,
    sigma_margin_to_LSL=4.2,
)
report_a = evaluate_deferral(state_a, defer_horizon=5)
print(report_a)

### 6.2 Scenario B — Drifting late-cycle (expect PROCEED)

In [ ]:
state_b = ToolState(
    days_since_last_pm=40,
    scheduled_pm_interval=45,
    recent_quality_index=1.45,
    spc_violations_last_7d=0,
    drift_slope_per_day=0.025,
    sigma_margin_to_USL=2.1,
    sigma_margin_to_LSL=3.8,
)
report_b = evaluate_deferral(state_b, defer_horizon=5)
print(report_b)

### 6.3 Scenario C — SPC violation (expect EMERGENCY)

In [ ]:
state_c = ToolState(
    days_since_last_pm=22,
    scheduled_pm_interval=45,
    recent_quality_index=1.51,
    spc_violations_last_7d=1,         # one WE violation triggers hard stop
    drift_slope_per_day=0.030,
    sigma_margin_to_USL=2.0,
    sigma_margin_to_LSL=3.5,
)
report_c = evaluate_deferral(state_c, defer_horizon=5)
print(report_c)

## 7. Tuning via Configuration

All decision thresholds and economic parameters are exposed via `DeferralConfig`. In production, these would load from YAML. Here we show how a more cost-tolerant fab (lower excursion cost → more eager to defer) changes the decision.

In [ ]:
# Aggressive cost-saving config: half the excursion cost assumption
cfg_aggressive = DeferralConfig(
    risk_threshold_defer=0.45,        # was 0.35
    cost_excursion=75_000.0,          # was 150_000
)

report_b2 = evaluate_deferral(state_b, defer_horizon=5, config=cfg_aggressive)
print('Scenario B re-evaluated with aggressive config:')
print(report_b2)

## 8. Summary & Talking Points

**Pipeline demonstrated**:

| Stage | Module | Output |
|---|---|---|
| Data generation | `data_generator` | 60 × 5 × 49 hierarchical Tox dataset |
| Wafer-level QC | `wafer_map` | 49-site contour map + WIW stats + quality_margin_index_WIW |
| Lot-level SPC | `oxide_spc` | X̄-S charts + WE rule violations + quality margin index |
| Trend monitoring | `oxide_spc` | 4-panel dashboard with rolling quality margin index |
| Decision support | `pm_deferral` | DEFER / PROCEED / EMERGENCY + rationale |

**Key engineering principles applied**:

- **Variation decomposition**: site / wafer / lot variation tracked separately (not pooled — sites are spatially correlated)
- **Multi-criteria decision**: hard stops always override economic optimization (safety first)
- **Conservative by default**: deferral only when risk < 35% AND net economic benefit > 0
- **Tunable**: all thresholds and costs exposed via config (no hardcoded constants in logic)

**What this would look like in production**:

- Daily cron pulls SECS/GEM telemetry + metro data → updates `ToolState`
- Engine outputs daily recommendation per toolset
- Engineer reviews high-risk cases; auto-approve LOW risk DEFER decisions
- Quarterly: re-fit Weibull β, η on PM history; retune cost params; calibrate risk weights